# 03.01 — Préparer l’apprentissage autonome (sans imitation)

Vérifie les chemins, les dépendances et les fichiers de circuit. Les contrôles réseau sont **sans conduite**. Aucun modèle ne s’entraîne à l’exécution de ce notebook.

In [12]:
from pathlib import Path
import sys
CWD=Path.cwd().resolve()
REPO=next((p for p in [CWD,*CWD.parents] if (p/'assetto_corsa_gym'/'assetto_corsa_gym'/'AssettoCorsaPlugin').is_dir()),None)
if REPO is None:
    raise FileNotFoundError('Ouvrir la racine IA_projet_Ouarti_ds_github ou un sous-dossier dans VS Code.')
PROJECT=REPO/'Code'
GYM_ROOT=REPO/'assetto_corsa_gym'/'assetto_corsa_gym'
assert PROJECT.is_dir(), 'Dossier Code introuvable'
if sys.version_info[:2] != (3,9):
    raise RuntimeError('Selectionner le kernel Python 3.9.13 dans VS Code. Actuel : '+sys.version)
for name in ['datasets','models','reports']:
    (PROJECT/name).mkdir(parents=True, exist_ok=True)
print('Depot :',REPO,'\nCode :',PROJECT,'\nPython :',sys.executable)

Depot : C:\Users\elkat\Documents\Github\IA_projet_Ouarti_ds_github 
Code : C:\Users\elkat\Documents\Github\IA_projet_Ouarti_ds_github\Code 
Python : c:\Users\elkat\AppData\Local\Programs\Python\Python39\python.exe


In [13]:
import numpy as np, pandas as pd
from scipy.spatial import cKDTree

class TrackGeometry:
    def __init__(self,path):
        df=pd.read_csv(path)
        required=['left_border_x','left_border_y','right_border_x','right_border_y']
        if not set(required).issubset(df.columns):
            raise ValueError('CSV geometrie non compatible, colonnes attendues: '+str(required))
        left=df[['left_border_x','left_border_y']].to_numpy(dtype=np.float64)
        right=df[['right_border_x','right_border_y']].to_numpy(dtype=np.float64)
        self.center=(left+right)*.5
        self.width=np.linalg.norm(left-right,axis=1)
        self.delta=np.roll(self.center,-1,axis=0)-self.center
        self.segment=np.linalg.norm(self.delta,axis=1)
        if (self.segment>.001).sum()<10:
            raise ValueError('CSV de circuit invalide: pas assez de points')
        self.arc=np.concatenate(([0.],np.cumsum(self.segment)))
        self.length=float(self.arc[-1])
        self.tree=cKDTree(self.center)
    def project(self,raw):
        xy=np.asarray([float(raw['world_position_x']),float(raw['world_position_y'])],dtype=np.float64)
        _,j=self.tree.query(xy)
        j=int(j)
        forward=self.delta[j]/max(float(self.segment[j]),1e-8)
        signed=float(forward[0]*(xy[1]-self.center[j,1])-forward[1]*(xy[0]-self.center[j,0]))
        heading=float(np.arctan2(forward[1],forward[0]))
        return float(self.arc[j]),signed,float(self.width[j]),heading

def numerical_state(raw,track):
    arc,offset,width,heading=track.project(raw)
    diff=(float(raw.get('yaw',0))-heading+np.pi)%(2*np.pi)-np.pi
    vec=[float(raw.get('speed',0))/100.,float(raw.get('accelX',0))/3.,
         float(raw.get('accelY',0))/3.,float(raw.get('angular_velocity_y',0))/4.,
         float(np.clip(offset/max(width/2.,1.),-3,3)),np.sin(diff),np.cos(diff),
         float(raw.get('accStatus',0)),float(raw.get('brakeStatus',0))]
    return np.clip(np.asarray(vec,dtype=np.float32),-5,5),arc,offset,width

def find_track_file(track_name):
    # Les fichiers .pkl sont des grilles d'occupation; ici la recompense
    # utilise les CSV bord gauche/bord droit du depot assetto_corsa_gym.
    name=str(track_name)
    path=GYM_ROOT/'AssettoCorsaConfigs'/'tracks'/(name+'.csv')
    if not path.is_file():
        raise FileNotFoundError('CSV de geometrie introuvable : '+str(path))
    return path

In [14]:

import importlib.util
import sys
import json
from pathlib import Path

# Vérification des bibliothèques nécessaires
required = {
    'numpy': 'numpy',
    'pandas': 'pandas',
    'scipy': 'scipy',
    'cv2': 'opencv-python',
    'mss': 'mss',
    'win32event': 'pywin32',
    'gymnasium': 'gymnasium',
    'stable_baselines3': 'stable-baselines3',
    'torch': 'torch'
}

missing = [
    name for name in required
    if importlib.util.find_spec(name) is None
]

print(
    'Modules absents :',
    missing if missing else 'AUCUN'
)

if missing:
    print(
        'Installer les modules manquants avec Python 3.9.'
    )

# Vérification de vJoy
vjoy_path = Path(
    r'C:\Program Files\vJoy\x64\vJoyInterface.dll'
)

print('vJoy installé :', vjoy_path.is_file())

# Vérification des fichiers de géométrie
tracks_dir = GYM_ROOT / 'AssettoCorsaConfigs' / 'tracks'

files = sorted(tracks_dir.glob('*.csv'))

print('CSV géométrie trouvés :', len(files))

print(
    'Circuits disponibles :',
    [p.stem for p in files if '-racing_line' not in p.stem]
)

# Vérification des fichiers PKL
pkl_dir = (
    REPO / 'AssettoCorsaGymDataSet'
    / 'AssettoCorsaConfigs'
    / 'tracks'
)

pkl_files = sorted(pkl_dir.glob('*.pkl'))

print('Fichiers PKL existants :', len(pkl_files))


Modules absents : ['gymnasium', 'stable_baselines3']
Installer les modules manquants avec Python 3.9.
vJoy installé : True
CSV géométrie trouvés : 21
Circuits disponibles : ['china-gp', 'imola', 'indianapolis_sp', 'ks_barcelona-layout_gp', 'ks_laguna_seca', 'ks_red_bull_ring-layout_gp', 'ks_silverstone-gp', 'monaco', 'monza', 'thunderhill_raceway_park-twomilewest']
Fichiers PKL existants : 9


In [15]:
# Verification GEOMETRIE hors ligne, en choisissant Barcelone qui a deja ete teste.
TRACK_ATTENDU='ks_barcelona-layout_gp'
CAR_ATTENDUE='abarth500'
file=find_track_file(TRACK_ATTENDU)
track=TrackGeometry(file)
print('Circuit:',file.name,'longueur approximative:',round(track.length),'m')
print('Les geometries CSV servent a la recompense, PAS a imiter des tours humains.')

Circuit: ks_barcelona-layout_gp.csv longueur approximative: 4647 m
Les geometries CSV servent a la recompense, PAS a imiter des tours humains.


In [16]:
import socket, json, ast, time
HOST='127.0.0.1'; PORT_UDP=2345; PORT_TCP=2347

def management(command,timeout=4):
    if command not in ('get_config','get_static_info'):
        raise ValueError('Cette fonction attend une commande qui retourne des donnees.')
    with socket.create_connection((HOST,PORT_TCP),timeout=timeout) as s:
        s.settimeout(timeout)
        s.sendall(command.encode('ascii'))
        data=b''
        deadline=time.monotonic()+timeout
        while time.monotonic()<deadline and len(data)<1024*1024:
            try: chunk=s.recv(65536)
            except socket.timeout: break
            if not chunk: break
            data+=chunk
            try: return json.loads(data)
            except (ValueError,UnicodeError): pass
            try: return ast.literal_eval(data.decode('utf8'))
            except (ValueError,SyntaxError,UnicodeError): pass
    raise RuntimeError('Reponse management absente/incomplete pour '+command+': '+repr(data[:150]))

def request_reset():
    # Le protocole sensors_par NE renvoie AUCUNE reponse a la commande reset.
    with socket.create_connection((HOST,PORT_TCP),timeout=4) as s:
        s.settimeout(4)
        s.sendall(b'reset')
        # Fermer la connexion : le serveur traite la commande sans ACK.

def open_telemetry(timeout=6):
    sock=socket.socket(socket.AF_INET,socket.SOCK_DGRAM)
    sock.settimeout(.30)
    sock.connect((HOST,PORT_UDP))
    end=time.monotonic()+timeout
    try:
        while time.monotonic()<end:
            sock.send(b'connect')
            try: data=sock.recv(2**18)
            except socket.timeout: continue
            if data==b'identified': return sock
        raise TimeoutError('Le plugin UDP 2345 ne repond pas : lancer une session de conduite.')
    except Exception:
        sock.close()
        raise

def next_packet(sock,timeout=3,after_steps=None):
    deadline=time.monotonic()+timeout
    while time.monotonic()<deadline:
        try: data=sock.recv(2**18)
        except socket.timeout: continue
        if data==b'disconnect': raise ConnectionError('Connexion telemetrie interrompue')
        if data==b'identified': continue
        try: packet=json.loads(data)
        except (ValueError,UnicodeError): continue
        if not isinstance(packet,dict): continue
        if after_steps is not None and int(packet.get('steps',-1))<=int(after_steps): continue
        packet['_received_unix']=time.time()
        return packet
    raise TimeoutError('Aucun nouvel echantillon telemetrie dans le delai imparti.')

def close_telemetry(sock):
    if sock is None: return
    try: sock.send(b'disconnect')
    except OSError: pass
    sock.close()

In [17]:
VERIFIER_CONNEXION_JEU=False
if VERIFIER_CONNEXION_JEU:
    cfg=management('get_config');s=management('get_static_info')
    print('Plugin:',{k:cfg.get(k) for k in ['sampling_freq','ego_sampling_freq',
              'screen_capture_enable','vjoy_executed_by_server','color_mode']})
    print('Session:',{k:s.get(k) for k in ['TrackName','TrackConfiguration','CarName','autoShifterOn']})
    assert cfg.get('screen_capture_enable'), 'Activer la camera.'
    assert cfg.get('vjoy_executed_by_server') is False, 'Ce pack utilise vJoy local.'
    print('Controle sans conduite OK. Les essais d action restent a faire dans le PPO.')
else:
    print('Verification du jeu non executee. Aucune action envoyee.')

Verification du jeu non executee. Aucune action envoyee.


**Important :** le dernier test `06_Test_Camera.ipynb` validé sur ton PC n'est pas modifié.
Le protocole `reset` de `sensors_par` ne renvoie pas d'ACK. Le PPO l'envoie sans attendre de réponse.
L’apprentissage avec caméra a un coût de calcul important : une longue formation sur un jeu temps réel peut prendre beaucoup de temps.